In [ ]:
#!/usr/bin/env python3
"""
Scrape J-Archive showgame pages and store into SQLite.

Usage:
  python scrape_jarchive_to_sqlite.py --start 9142 --end 9242 --db jeopardy.db

Notes:
- Be polite: delay between requests; don't run this in parallel.
- This script does not bypass any bot protection. If blocked, it will fail.
"""

import argparse
import re
import sqlite3
import sys
import time
from dataclasses import dataclass
from typing import List, Optional, Tuple

import requests
from bs4 import BeautifulSoup

BASE_URL = "https://j-archive.com/showgame.php?game_id={game_id}"


def init_db(conn: sqlite3.Connection) -> None:
    conn.execute("PRAGMA journal_mode=WAL;")
    conn.execute("PRAGMA foreign_keys=ON;")

    conn.executescript(
        """
        CREATE TABLE IF NOT EXISTS episode (
          game_id     INTEGER PRIMARY KEY,
          show_number INTEGER NOT NULL,
          air_date    TEXT    NOT NULL,
          title       TEXT
        );

        CREATE TABLE IF NOT EXISTS category (
          category_id INTEGER PRIMARY KEY AUTOINCREMENT,
          game_id     INTEGER NOT NULL REFERENCES episode(game_id),
          round       TEXT    NOT NULL CHECK (round IN ('J','DJ','FJ')),
          board_col   INTEGER,
          name        TEXT    NOT NULL,
          comments    TEXT,
          UNIQUE(game_id, round, board_col)
        );

        CREATE TABLE IF NOT EXISTS clue (
          clue_id          INTEGER PRIMARY KEY AUTOINCREMENT,
          category_id      INTEGER NOT NULL REFERENCES category(category_id),
          row_in_category  INTEGER NOT NULL,
          value_cents      INTEGER,
          clue_order       INTEGER,
          is_daily_double  INTEGER NOT NULL DEFAULT 0,
          clue_text        TEXT NOT NULL,
          correct_response TEXT NOT NULL,
          UNIQUE(category_id, row_in_category)
        );

        CREATE TABLE IF NOT EXISTS used_category (
          category_id INTEGER PRIMARY KEY REFERENCES category(category_id),
          used_at     TEXT NOT NULL
        );

        CREATE INDEX IF NOT EXISTS idx_episode_air_date ON episode(air_date);
        CREATE INDEX IF NOT EXISTS idx_category_round_game ON category(round, game_id);
        CREATE INDEX IF NOT EXISTS idx_clue_category_row ON clue(category_id, row_in_category);
        """
    )


def parse_air_date(title_text: str) -> Tuple[int, str]:
    """
    Supports both:
      - "Show #9365 - Friday, June 27, 2025"
      - "Show #9514, aired 2026-03-05" (from <title>)
    Returns: (show_number, air_date_iso)
    """
    s = title_text.strip()

    import datetime as dt

    m = re.search(r"Show\s+#(\d+)\s+-\s+(.+)$", s)
    if m:
        show_number = int(m.group(1))
        date_str = re.sub(r"^[A-Za-z]+,\s+", "", m.group(2)).strip()
        dt_obj = dt.datetime.strptime(date_str, "%B %d, %Y")
        return show_number, dt_obj.date().isoformat()

    m = re.search(r"Show\s+#(\d+),\s+aired\s+(\d{4}-\d{2}-\d{2})$", s)
    if m:
        show_number = int(m.group(1))
        dt_obj = dt.datetime.strptime(m.group(2), "%Y-%m-%d")
        return show_number, dt_obj.date().isoformat()

    raise ValueError(f"Could not parse show number/date from: {title_text!r}")


def clean_text(el) -> str:
    """Extract plain text from an element, collapsing whitespace and removing link markup."""
    if el is None:
        return ""
    txt = el.get_text(" ", strip=True)
    # Collapse multiple spaces
    txt = re.sub(r"\s+", " ", txt).strip()
    return txt


def parse_money_to_cents(s: str) -> Optional[int]:
    """
    "$200" -> 20000
    "DD: $1,000" -> 100000 (still store actual face value; mark DD separately)
    Return None if not parseable (e.g., Final Jeopardy).
    """
    if not s:
        return None
    m = re.search(r"\$([\d,]+)", s)
    if not m:
        return None
    dollars = int(m.group(1).replace(",", ""))
    return dollars * 100


def is_daily_double_value_cell(clue_header_table: BeautifulSoup) -> int:
    """
    Daily double shows td class="clue_value_daily_double" with text like "DD: $1,000"
    """
    if clue_header_table is None:
        return 0
    td = clue_header_table.select_one("td.clue_value_daily_double")
    return 1 if td else 0


def fetch_soup(session: requests.Session, url: str, timeout: int = 30) -> Optional[BeautifulSoup]:
    last_error: Optional[Exception] = None
    for attempt in range(3):
        try:
            resp = session.get(url, timeout=timeout)
            if resp.status_code == 404:
                return None
            resp.raise_for_status()
            try:
                return BeautifulSoup(resp.text, "lxml")
            except Exception:
                return BeautifulSoup(resp.text, "html.parser")
        except requests.RequestException as e:
            last_error = e
            if attempt < 2:
                time.sleep(1.5 * (attempt + 1))
                continue
            raise
    if last_error:
        raise last_error
    return None


@dataclass
class ParsedClue:
    round_code: str       # 'J' / 'DJ' / 'FJ'
    col: Optional[int]    # 1..6, None for FJ
    row: int              # 1..5 for J/DJ, 1 for FJ
    value_cents: Optional[int]
    clue_order: Optional[int]
    is_dd: int
    clue_text: str
    correct_response: str


def parse_round_board(round_div: BeautifulSoup, round_code: str) -> Tuple[List[str], List[ParsedClue]]:
    """
    For J/DJ: returns (categories_by_col, clues)
    categories_by_col: list length 6, index 0..5 => col 1..6
    clues: 30 clues with col,row mapping from ids like clue_J_4_5
    """
    board = round_div.select_one("table.round")
    if board is None:
        return [], []

    # Categories: first <tr> has 6 <td class="category"> each with <td class="category_name">
    cat_names = []
    cat_row = board.select_one("tr")
    cat_cells = cat_row.select("td.category")
    for td in cat_cells:
        name_el = td.select_one(".category_name")
        cat_names.append(clean_text(name_el))

    # Clues: find all td.clue and parse clue ids for col/row
    clues: List[ParsedClue] = []
    for clue_td in board.select("td.clue"):
        clue_text_el = clue_td.select_one("td.clue_text[id^=clue_{}_]".format(round_code))
        if clue_text_el is None:
            # Sometimes missing clues exist; skip if no standard clue cell
            continue

        clue_id = clue_text_el.get("id", "")
        # clue_id like "clue_J_5_2" or "clue_DJ_3_4"
        m = re.match(r"clue_(J|DJ)_(\d+)_(\d+)", clue_id)
        if not m:
            continue
        _rc, col_s, row_s = m.groups()
        col = int(col_s)
        row = int(row_s)

        # correct response in the paired hidden cell id with suffix "_r"
        resp_el = clue_td.select_one(f"td.clue_text#clue_{round_code}_{col}_{row}_r em.correct_response")
        correct_response = clean_text(resp_el)

        clue_text = clean_text(clue_text_el)

        header = clue_td.select_one("table.clue_header")
        value_el = header.select_one("td.clue_value, td.clue_value_daily_double")
        value_cents = parse_money_to_cents(clean_text(value_el))

        order_el = header.select_one("td.clue_order_number a")
        clue_order = None
        if order_el:
            try:
                clue_order = int(clean_text(order_el))
            except ValueError:
                clue_order = None

        dd_flag = is_daily_double_value_cell(header)

        # Some correct responses include italics via <i>; clean_text handles it.
        if correct_response and clue_text:
            clues.append(
                ParsedClue(
                    round_code=round_code,
                    col=col,
                    row=row,
                    value_cents=value_cents,
                    clue_order=clue_order,
                    is_dd=dd_flag,
                    clue_text=clue_text,
                    correct_response=correct_response,
                )
            )

    return cat_names, clues


def parse_final_round(final_div: BeautifulSoup) -> Tuple[str, ParsedClue]:
    cat_el = final_div.select_one("td.category_name")
    category = clean_text(cat_el)

    clue_text_el = final_div.select_one("td#clue_FJ.clue_text")
    clue_text = clean_text(clue_text_el)

    resp_el = final_div.select_one("td#clue_FJ_r em.correct_response")
    correct_response = clean_text(resp_el)

    parsed = ParsedClue(
        round_code="FJ",
        col=None,
        row=1,
        value_cents=None,
        clue_order=None,
        is_dd=0,
        clue_text=clue_text,
        correct_response=correct_response,
    )
    return category, parsed


def upsert_episode(conn: sqlite3.Connection, game_id: int, show_number: int, air_date: str, title: str) -> None:
    conn.execute(
        """
        INSERT INTO episode (game_id, show_number, air_date, title)
        VALUES (?, ?, ?, ?)
        ON CONFLICT(game_id) DO UPDATE SET
          show_number=excluded.show_number,
          air_date=excluded.air_date,
          title=excluded.title
        """,
        (game_id, show_number, air_date, title),
    )


def get_or_create_category(
    conn: sqlite3.Connection,
    game_id: int,
    round_code: str,
    board_col: Optional[int],
    name: str,
    comments: str = "",
) -> int:
    conn.execute(
        """
        INSERT OR IGNORE INTO category (game_id, round, board_col, name, comments)
        VALUES (?, ?, ?, ?, ?)
        """,
        (game_id, round_code, board_col, name, comments),
    )
    row = conn.execute(
        """
        SELECT category_id FROM category
        WHERE game_id=? AND round=? AND (board_col IS ? OR board_col=?)
        """,
        (game_id, round_code, board_col, board_col),
    ).fetchone()
    if not row:
        raise RuntimeError("Failed to create/find category row")
    return int(row[0])


def insert_clue(conn: sqlite3.Connection, category_id: int, c: ParsedClue) -> None:
    conn.execute(
        """
        INSERT OR REPLACE INTO clue
          (category_id, row_in_category, value_cents, clue_order, is_daily_double, clue_text, correct_response)
        VALUES (?, ?, ?, ?, ?, ?, ?)
        """,
        (
            category_id,
            c.row,
            c.value_cents,
            c.clue_order,
            c.is_dd,
            c.clue_text,
            c.correct_response,
        ),
    )


def scrape_game(session: requests.Session, conn: sqlite3.Connection, game_id: int) -> bool:
    url = BASE_URL.format(game_id=game_id)
    soup = fetch_soup(session, url)
    if soup is None:
        print(f"[{game_id}] 404 not found; skipping")
        return False

    h1 = soup.select_one("#game_title h1")
    title_text = clean_text(h1)
    if not title_text:
        # Fallback: some responses omit #game_title but still provide a parseable page title.
        page_title = clean_text(soup.select_one("title"))
        title_text = re.sub(r"^J!\s*Archive\s+-\s+", "", page_title)
    if not title_text:
        print(f"[{game_id}] Missing game title and <title>; skipping")
        return False
    try:
        show_number, air_date = parse_air_date(title_text)
    except Exception as e:
        print(f"[{game_id}] Failed parsing title: {e}; skipping")
        return False

    upsert_episode(conn, game_id, show_number, air_date, title_text)

    # Jeopardy + Double Jeopardy
    j_div = soup.select_one("#jeopardy_round")
    dj_div = soup.select_one("#double_jeopardy_round")
    fj_div = soup.select_one("#final_jeopardy_round")

    if j_div:
        cat_names, clues = parse_round_board(j_div, "J")
        for col_idx, cat_name in enumerate(cat_names, start=1):
            cat_id = get_or_create_category(conn, game_id, "J", col_idx, cat_name)
            # Insert clues for that column
            for c in clues:
                if c.col == col_idx:
                    insert_clue(conn, cat_id, c)

    if dj_div:
        cat_names, clues = parse_round_board(dj_div, "DJ")
        for col_idx, cat_name in enumerate(cat_names, start=1):
            cat_id = get_or_create_category(conn, game_id, "DJ", col_idx, cat_name)
            for c in clues:
                if c.col == col_idx:
                    insert_clue(conn, cat_id, c)

    if fj_div:
        fj_cat, fj_clue = parse_final_round(fj_div)
        cat_id = get_or_create_category(conn, game_id, "FJ", 0, fj_cat)  # board_col=0 for FJ
        insert_clue(conn, cat_id, fj_clue)

    return True


def main(argv: Optional[List[str]] = None) -> None:
    ap = argparse.ArgumentParser()
    ap.add_argument("--start", type=int, required=True)
    ap.add_argument("--end", type=int, required=True)
    ap.add_argument("--db", type=str, default="jeopardy.db")
    ap.add_argument("--delay", type=float, default=1.25, help="Seconds to sleep between requests")
    ap.add_argument("--timeout", type=int, default=30)
    args = ap.parse_args(argv)

    conn = sqlite3.connect(args.db)
    init_db(conn)

    session = requests.Session()
    session.headers.update(
        {
            "User-Agent": "Mozilla/5.0 (compatible; personal-jeopardy-scraper/1.0; +https://example.invalid)",
            "Accept-Language": "en-US,en;q=0.9",
        }
    )

    ok = 0
    skipped = 0

    for game_id in range(args.start, args.end + 1):
        try:
            with conn:
                success = scrape_game(session, conn, game_id)
            if success:
                ok += 1
                print(f"[{game_id}] OK")
            else:
                skipped += 1
        except requests.HTTPError as e:
            skipped += 1
            print(f"[{game_id}] HTTP error: {e}")
        except Exception as e:
            skipped += 1
            print(f"[{game_id}] Error: {e}")
        time.sleep(args.delay)

    print(f"Done. Inserted/updated games: {ok}, skipped: {skipped}. DB: {args.db}")


if __name__ == "__main__":
    if "ipykernel" in sys.argv[0]:
        main(["--start", "9146", "--end", "9513"])
    else:
        main()

In [ ]:
!pip install lxml